In [ ]:
import os
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer


In [ ]:
# rutas base
REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.today().strftime('%Y%m%d')
INPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '03_normalizacion_v2' / RUN_ID
OUTPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '04_featuring_v2' / RUN_ID
REPORTS_DIR = REPO_ROOT / 'reports' / '04_featuring_v2' / RUN_ID
TARGET_DIRS = ['1_vs_resto', '5_vs_resto']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_DIR.exists():
    candidates = sorted((REPO_ROOT / 'data' / 'intermediate' / '03_normalizacion_v2').glob('*/1_vs_resto'))
    if candidates:
        INPUT_DIR = candidates[-1].parent
        print(f'Usando ultima normalizacion encontrada: {INPUT_DIR}')
    else:
        raise FileNotFoundError('No se encontro normalizacion en data/intermediate/03_normalizacion_v2')

print('Rutas configuradas:')
print(f'INPUT_DIR: {INPUT_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'REPORTS_DIR: {REPORTS_DIR}')
print(f'TARGET_DIRS: {TARGET_DIRS}')


In [ ]:
def add_features(df):
    df_extended = df.copy().astype(np.float32)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if df[col].nunique() > 2]

    detalle_columnas = []

    for col in numeric_cols:
        try:
            col_data = df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

            # Log, sqrt, squared
            df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_log", "log"))

            df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_sqrt", "sqrt"))

            df_extended[f"{col}_squared"] = col_data ** 2
            detalle_columnas.append((col, f"{col}_squared", "squared"))

            # Yeo-Johnson
            pt_yj = PowerTransformer(method="yeo-johnson")
            yj_data = pt_yj.fit_transform(col_data.values.reshape(-1, 1)).flatten()
            df_extended[f"{col}_yeojohnson"] = yj_data
            detalle_columnas.append((col, f"{col}_yeojohnson", "yeojohnson"))

            # Box-Cox (solo si todos los valores son > 0)
            if (col_data > 0).all():
                pt_bc = PowerTransformer(method="box-cox")
                bc_data = pt_bc.fit_transform(col_data.values.reshape(-1, 1)).flatten()
                df_extended[f"{col}_boxcox"] = bc_data
                detalle_columnas.append((col, f"{col}_boxcox", "boxcox"))

        except Exception as e:
            print(f"Error en columna {col}: {e}")

    return df_extended, pd.DataFrame(detalle_columnas, columns=["original", "generada", "tipo"])


# Procesar cada target y consolidar metricas por normalizacion
resumen_global = []

for target_name in TARGET_DIRS:
    target_input_dir = INPUT_DIR / target_name
    if not target_input_dir.exists():
        print(f"No se encontro rama {target_name}; se omite.")
        continue

    target_output_dir = OUTPUT_DIR / target_name
    target_reports_dir = REPORTS_DIR / target_name
    metric_dir = target_reports_dir / 'metricas_generacion'

    target_output_dir.mkdir(parents=True, exist_ok=True)
    target_reports_dir.mkdir(parents=True, exist_ok=True)
    metric_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcesando target: {target_name}")

    resumen_norms = []

    for norm_name in sorted(os.listdir(target_input_dir)):
        norm_dir = target_input_dir / norm_name
        if not norm_dir.is_dir():
            continue

        out_dir = target_output_dir / norm_name
        out_dir.mkdir(parents=True, exist_ok=True)

        try:
            X_train = pd.read_parquet(norm_dir / f"X_train_{norm_name}.parquet")
            X_test = pd.read_parquet(norm_dir / f"X_test_{norm_name}.parquet")
            y_train = pd.read_parquet(norm_dir / f"y_train_{norm_name}.parquet")
            y_test = pd.read_parquet(norm_dir / f"y_test_{norm_name}.parquet")
        except Exception as e:
            print(f"Error leyendo archivos en {target_name}/{norm_name}: {e}")
            continue

        # Guardar version original
        X_train.to_parquet(out_dir / f"X_train_{norm_name}_ORIGINAL.parquet")
        X_test.to_parquet(out_dir / f"X_test_{norm_name}_ORIGINAL.parquet")
        y_train.to_parquet(out_dir / f"y_train_{norm_name}_ORIGINAL.parquet")
        y_test.to_parquet(out_dir / f"y_test_{norm_name}_ORIGINAL.parquet")

        # Aplicar feature engineering SOLO en train
        X_train_fe, df_detalle = add_features(X_train)

        # Replicar las columnas generadas en test
        X_test_fe = X_test.copy()
        for _, row in df_detalle.iterrows():
            col = row["generada"]
            base_col = row["original"]
            tipo = row["tipo"]
            try:
                base_data = X_test[base_col].replace([np.inf, -np.inf], np.nan).fillna(0)
                if tipo == "log":
                    X_test_fe[col] = np.log1p(np.abs(base_data))
                elif tipo == "sqrt":
                    X_test_fe[col] = np.sqrt(np.abs(base_data))
                elif tipo == "squared":
                    X_test_fe[col] = base_data ** 2
                elif tipo == "yeojohnson":
                    pt_yj = PowerTransformer(method="yeo-johnson")
                    X_test_fe[col] = pt_yj.fit(
                        X_train[[base_col]].replace([np.inf, -np.inf], np.nan).fillna(0)
                    ).transform(base_data.values.reshape(-1, 1)).flatten()
                elif tipo == "boxcox":
                    if (base_data > 0).all():
                        pt_bc = PowerTransformer(method="box-cox")
                        X_test_fe[col] = pt_bc.fit(
                            X_train[[base_col]].replace([np.inf, -np.inf], np.nan).fillna(0)
                        ).transform(base_data.values.reshape(-1, 1)).flatten()
            except Exception as e:
                print(f"Error generando {col} en test: {e}")

        # Asegurar mismo orden de columnas
        X_test_fe = X_test_fe[X_train_fe.columns]

        # Guardar nuevos datasets
        X_train_fe.to_parquet(out_dir / f"X_train_{norm_name}_FE.parquet")
        X_test_fe.to_parquet(out_dir / f"X_test_{norm_name}_FE.parquet")
        y_train.to_parquet(out_dir / f"y_train_{norm_name}_FE.parquet")
        y_test.to_parquet(out_dir / f"y_test_{norm_name}_FE.parquet")

        # Guardar CSV con detalle por normalizacion
        detalle_file = metric_dir / f"detalle_columnas_{norm_name}.csv"
        df_detalle.to_csv(detalle_file, index=False)

        resumen_tipos = df_detalle["tipo"].value_counts().to_dict()
        fila = {
            "target": target_name,
            "normalizacion": norm_name,
            "columnas_originales": int(X_train.shape[1]),
            "nuevas_columnas_generadas": int(df_detalle.shape[0]),
            "total_final_columnas": int(X_train_fe.shape[1]),
        }
        for tipo, count in resumen_tipos.items():
            fila[f"tipo_{tipo}"] = int(count)
        resumen_norms.append(fila)
        resumen_global.append(fila)

        # Consola por cada variante
        print(f"\n[{target_name} | {norm_name}] Ingenieria aplicada")
        print(f"Columnas originales: {X_train.shape[1]}")
        print(f"Nuevas columnas generadas: {df_detalle.shape[0]}")
        print(f"Total final de columnas: {X_train_fe.shape[1]}")
        print("Detalle por tipo de transformacion:")
        for tipo, count in resumen_tipos.items():
            print(f"  {tipo}: {count}")
        print(f"CSV guardado en: {detalle_file}")

    if resumen_norms:
        df_resumen_norms = pd.DataFrame(resumen_norms).sort_values("normalizacion").fillna(0)
        resumen_file = metric_dir / "resumen_columnas_por_normalizacion.csv"
        df_resumen_norms.to_csv(resumen_file, index=False)
        print(f"\nResumen consolidado guardado en: {resumen_file}")
        display(df_resumen_norms)
    else:
        print(f"No se encontraron normalizaciones para procesar en {target_name}.")

if resumen_global:
    df_resumen_global = pd.DataFrame(resumen_global).sort_values(["target", "normalizacion"]).fillna(0)
    resumen_global_file = REPORTS_DIR / "resumen_columnas_por_normalizacion_global.csv"
    df_resumen_global.to_csv(resumen_global_file, index=False)
    print(f"\nResumen global guardado en: {resumen_global_file}")
else:
    print("No se encontraron targets con normalizaciones para procesar.")


In [ ]:
# Visualizacion rapida del resumen global
if "df_resumen_global" in globals():
    display(df_resumen_global)
else:
    resumen_global_file = REPORTS_DIR / "resumen_columnas_por_normalizacion_global.csv"
    if resumen_global_file.exists():
        display(pd.read_csv(resumen_global_file))
    else:
        print("Primero ejecuta la celda principal de featuring para generar el resumen.")
